In [1]:
import os

In [2]:
from docling.document_converter import DocumentConverter,PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
from docling.datamodel.pipeline_options import PdfPipelineOptions
import tiktoken
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.http.exceptions import UnexpectedResponse

c:\dev\lv-assignment1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
embedding_model = "text-embedding-3-small"
dimension=1536

In [65]:
hr_ploicy_file=r"C:\dev\lv-assignment1\zdata\ABC_Corporation_HR_Policy_Manual.pdf"
law_policy_file=r"C:\dev\lv-assignment1\zdata\Historical_Court_Verdicts.pdf"

In [66]:
def parse_document(file_path:str):

    pdf_option = PdfPipelineOptions(
    do_ocr=False,
    do_table_structure=True)
    converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_option)})
    doc=converter.convert(file_path)
    return doc.document

hr_policy_doc=parse_document(hr_ploicy_file)
law_policy_doc=parse_document(law_policy_file)

In [8]:
def chunking(document):
    tiktoken_encoder = tiktoken.get_encoding("cl100k_base")
    tokenizer = OpenAITokenizer(tokenizer=tiktoken_encoder,max_tokens=512)
    chunker = HybridChunker(tokenizer=tokenizer, chunk_size=512, chunk_overlap=50,merge_peers=True)
    raw_chunks = list(chunker.chunk(document))
    return raw_chunks

In [15]:
def data_prep(chunks):

    """we are preparing data in the format required by VB"""
    result = []

    for idx,chunk in enumerate(chunks):
        headings=[]
        if chunk.meta.headings:
            headings=chunk.meta.headings
        page_no =[]
        if chunk.meta.doc_items:
            for item in chunk.meta.doc_items:
                for prov in item.prov:
                    page_no.append(prov.page_no)
        captions=[]
        if chunk.meta.captions:
            captions=chunk.meta.captions

        result.append(
            {
                "filename":chunk.meta.origin.filename,
                "headings":headings,
                "page_no":page_no,
                "captions":captions,
                "text":chunk.text
            }
        )
    return result



In [16]:
def embedding(chucks_cleaned):
    embedding_client = OpenAI(api_key=openai_api_key)
    embedding =[]
    for chunk in chucks_cleaned:
        response = embedding_client.embeddings.create(
        input=chunk["text"],
        model=embedding_model,
        encoding_format="float")
        embedding.append(response.data[0].embedding)
    return embedding

Pinecone Index creation and addition work

In [35]:
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_environment = os.getenv("PINECONE_ENVIRONMENT")
pinecone_index_name = os.getenv("PINECONE_INDEX_NAME")


In [36]:
pinecone_index_name

'hr-policy'

In [37]:
from pinecone.grpc import PineconeGRPC

In [38]:
pc=PineconeGRPC(api_key=pinecone_api_key)

In [39]:
index_description = pc.describe_index(name=pinecone_index_name)

In [40]:
index_description.host

'hr-policy-rev8dya.svc.aped-4627-b74a.pinecone.io'

In [41]:
index = pc.Index(host=index_description.host)

In [133]:
for chunk, embed in zip(result, embedding):
    index.upsert(
        vectors=[
            {
                "id": f"{chunk['filename']}_{chunk['page_no']}_{chunk['headings']}",
                "values": embed,
                "metadata": {
                    "filename": chunk["filename"],
                    "headings": chunk["headings"],
                    "page_no": int(chunk["page_no"][0]) if isinstance(chunk["page_no"], list) else int(chunk["page_no"]),
                    "captions": chunk["captions"],
                    "text": chunk["text"]
                }
            }
        ]
    )

In [31]:
def HR_retrieval(question):
    embedding_client = OpenAI(api_key=openai_api_key)
    query_emded=embedding_client.embeddings.create(
        input=question,
        model=embedding_model,
        encoding_format="float"
    )
    query_vector=query_emded.data[0].embedding
    answer = index.query(
        vector=query_vector,
        top_k=2,
        include_metadata=True
    )
    return answer.matches[0].metadata["text"]

In [42]:
HR_retrieval("when is the performance review done?")

'Performance reviews are conducted bi n annually. Employees receive structured feedback, goal alignment, and development planning. Outstanding performers may receive incentives and career advancement opportunities.'

Qdrant work

In [67]:
raw_chunks=chunking(law_policy_doc)

In [68]:
raw_chunks

[DocChunk(text="This landmark decision by the United States Supreme Court declared state laws establishing separate public schools for Black and white students unconstitutional. The Court held that segregation in public education violated the Equal Protection Clause of the Fourteenth Amendment. The ruling overturned the precedent set by Plessy v. Ferguson (1896), which had allowed 'separate but equal' facilities. The verdict became a cornerstone of the Civil Rights Movement and reshaped American constitutional law regarding equality and discrimination.", meta=DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[DocItem(self_ref='#/texts/1', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=78.0, t=712.1059705078125, r=576.0, b=605.0059705078124, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 542))

In [80]:
clean_data=data_prep(raw_chunks)

C:\Users\ACER\AppData\Local\Temp\ipykernel_13292\3104883512.py:16: DeprecationWarning: deprecated
  if chunk.meta.captions:


In [82]:
clean_data

[{'filename': 'Historical_Court_Verdicts.pdf',
  'headings': ['Case 1: Brown v. Board of Education (1954)'],
  'page_no': [1],
  'captions': [],
  'text': "This landmark decision by the United States Supreme Court declared state laws establishing separate public schools for Black and white students unconstitutional. The Court held that segregation in public education violated the Equal Protection Clause of the Fourteenth Amendment. The ruling overturned the precedent set by Plessy v. Ferguson (1896), which had allowed 'separate but equal' facilities. The verdict became a cornerstone of the Civil Rights Movement and reshaped American constitutional law regarding equality and discrimination."},
 {'filename': 'Historical_Court_Verdicts.pdf',
  'headings': ['Case 2: Roe v. Wade (1973)'],
  'page_no': [2],
  'captions': [],
  'text': "The Supreme Court ruled that a woman's right to choose an abortion fell within the right to privacy protected by the Fourteenth Amendment. The decision invali

In [81]:
embed=embedding(clean_data)

In [20]:
qdrant_url=os.getenv("Qdrant_host")

In [22]:
qdrant_api_key = os.getenv("Qdrant_api_key")

In [25]:
qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key)

In [26]:
qdrant_client

collection is created in the cluster

In [28]:
qdrant_client.create_collection(
    collection_name="law_policy_collection",
    vectors_config= VectorParams(
        size=1536, distance=Distance.COSINE)
)

True

Upsert data into the qdrant client

In [30]:
import uuid

In [83]:
qdrant_client.upsert(
    collection_name="law_policy_collection",
    points=[
        {
            "id": str(uuid.uuid4()),
            "vector": embed,
            "payload": {
                "filename": chunk["filename"],
                "headings": chunk["headings"],
                "page_no": int(chunk["page_no"][0]) if isinstance(chunk["page_no"], list) else int(chunk["page_no"]),
                "captions": chunk["captions"],
                "text": chunk["text"]
            }
        }
        for chunk, embed in zip(clean_data, embed)
    ]
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [93]:
def law_retrieval(question):
    embedding_client = OpenAI(api_key=openai_api_key)
    query_emded=embedding_client.embeddings.create(
        input=question,
        model=embedding_model,
        encoding_format="float"
    )
    query_vector=query_emded.data[0].embedding
    answer=qdrant_client.query_points(
        query=query_vector,
        limit=3,
        collection_name="law_policy_collection"
    )
    return answer.points[0].payload["text"]

In [94]:
law_retrieval("what is Miranda case about?")

"In this decision, the Supreme Court held that detained criminal suspects must be informed of their constitutional rights prior to police interrogation. These rights include the right to remain silent and the right to an attorney. The ruling aimed to protect individuals from self-incrimination under the Fifth Amendment. As a result, the 'Miranda rights' warning became a standard practice in law enforcement procedures nationwide."

code to delete the vectors

In [ ]:
from qdrant_client import models
qdrant_client.delete(
    collection_name="law_policy_collection",
    points_selector=models.FilterSelector(
        filter=models.Filter()
    ),
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)